In [1]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 

PROJECT_ROOT = Path.cwd() 

if PROJECT_ROOT.name == "notebooks": 
    PROJECT_ROOT = PROJECT_ROOT.parent 

sys.path.insert(0, str(PROJECT_ROOT))

from src.vector_store import VectorStore 

embeddings = np.load(PROJECT_ROOT / "data/embeddings/abo_image_embeddings.npy")

metadata = pd.read_csv(PROJECT_ROOT / "data/embeddings/abo_image_metadata.csv") 

print(embeddings.shape) 
print(len(metadata))

(1000, 512)
1000


In [2]:
store_path = PROJECT_ROOT / "qdrant_storage" 

vector_store = VectorStore(
    storage_path=str(store_path), 
    collection_name="abo_products", 
    vector_size=embeddings.shape[1], 
)

In [3]:
if vector_store.client.collection_exists("abo_products"): 
    vector_store.client.delete_collection("abo_products") 

vector_store.create_collection() 
print("Collection created.") 

Collection created.


In [4]:
vector_store.index_embeddings(
    embeddings=embeddings, 
    metadata=metadata, 
)

print("Stored points:", vector_store.count()) 

Stored points: 1000


In [7]:
vector_store.client.close() 

reloaded_store = VectorStore(
    storage_path=str(store_path),
    collection_name="abo_products",
    vector_size=embeddings.shape[1],
)

print("Reloaded count:", reloaded_store.count())

Reloaded count: 1000


In [8]:
# Testing resuable store 

from src.embedder import Embedder 

embedder = Embedder() 
query_embedding = embedder.embed_text("a black handbag") 

results = reloaded_store.search(
    query_embedding=query_embedding, 
    limit=5, 
)

for result in results: 
    print(result) 

/Users/aob/AOB/CS/Projects/clip-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 19:38:28.066000 18371 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 79077.83it/s]


id=941 version=0 score=0.30643937277470185 payload={'item_id': 'B081HMLLBR', 'image_id': '71HQLq+AUzL', 'product_type': 'HANDBAG', 'title': "Nia & Nicole Women's Handbag (Navy Blue)", 'image_path': 'data/ABO/abo-images-small/images/small/0f/0fa0e93d.jpg'} vector=None shard_key=None order_value=None
id=37 version=0 score=0.3048613825035017 payload={'item_id': 'B000R5H79K', 'image_id': '41pxABUoPoL', 'product_type': 'CELLULAR_PHONE_CASE', 'title': 'Rivet The Wrap Fashion Case for Mobile Phones', 'image_path': 'data/ABO/abo-images-small/images/small/5c/5c4c18c9.jpg'} vector=None shard_key=None order_value=None
id=952 version=0 score=0.30163453555614606 payload={'item_id': 'B07WRJFTRK', 'image_id': '71Ps6Eena3L', 'product_type': 'HANDBAG', 'title': "Flavia Women's Handbag (Black)", 'image_path': 'data/ABO/abo-images-small/images/small/c4/c4270338.jpg'} vector=None shard_key=None order_value=None
id=912 version=0 score=0.29639846201502773 payload={'item_id': 'B06X17B2N4', 'image_id': '61tH8

In [9]:
assert len(results) == 5
assert results[0]["score"] >= results[-1]["score"]
assert "image_path" in results[0]
assert "product_type" in results[0]

TypeError: 'ScoredPoint' object is not subscriptable